# Phase 2.5 — Germination-candidate segmentation validation

This workflow evaluates interpretable **candidate** masks. It does not identify individual seeds, label radicles, extract features, or begin Phase 3.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

current_directory = Path.cwd().resolve()
candidates = [current_directory, *current_directory.parents, current_directory / 'seed-germination']
PROJECT_ROOT = next((path for path in candidates if (path / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not find the project root containing src/.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ProjectPaths
from src.germination_validation import CandidateMaskConfig, run_phase_2_5_validation
from src.io_utils import build_inventory

paths = ProjectPaths(root=PROJECT_ROOT)
inventory_path = paths.tables / 'dataset_inventory.csv'
inventory = (pd.read_csv(inventory_path, parse_dates=['timestamp']) if inventory_path.exists()
             else build_inventory(paths.raw_images))
inventory = inventory.loc[inventory['is_readable']].copy()
len(inventory)

In [ ]:
# Adjust only after reviewing outputs; no thresholds are biological definitions.
candidate_config = CandidateMaskConfig(
    lab_b_min=145, lab_a_max=140, lightness_min=135, brightness_min=175,
    hsv_hue_min=12, hsv_hue_max=48, hsv_saturation_min=35, hsv_value_min=110,
    exclude_seed_body=True, restrict_to_boundary_band=True, boundary_kernel_size=31,
)

figure_dir = paths.figures / 'germination_validation'
table_dir = paths.tables / 'germination_validation'
representatives, summary, recommendation = run_phase_2_5_validation(
    inventory, figure_dir, table_dir, candidate_config
)
representatives[['representative_period', 'filename', 'timestamp']]

In [ ]:
summary.sort_values(['representative_period', 'method'])

In [ ]:
recommendation

## Required review before Phase 3

Open the three diagnostic overlays in `results/figures/germination_validation/`. Decide whether highlighted regions align with visible emerging structures and whether the boundary-band restriction omits relevant regions. Until that evidence is recorded, the recommendation intentionally keeps `ready_for_phase_3` as `false`.